In [17]:
pip install pandas scikit-learn numpy streamlit

In [18]:
import pandas as pd

# Load the uploaded file
df = pd.read_csv("/content/movies.csv")

# Keep relevant columns
df = df[['title', 'genres']]

# Clean data
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df['genres'] = df['genres'].str.replace('|', ' ', regex=False)

# Save cleaned version
df.to_csv("/content/clean_movies.csv", index=False)

print("✅ Cleaned dataset sample:")
df.head()


✅ Cleaned dataset sample:


,title,genres
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy Romance
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy


In [19]:
from google.colab import files
files.download("/content/clean_movies.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# Step 0 — Install fuzzywuzzy

!pip install fuzzywuzzy[speedup]


# Step 1 — Load dataset

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from fuzzywuzzy import process
import re

# Load CSV
movies = pd.read_csv("/content/clean_movies.csv")


# TF-IDF vectorization
tfidf = TfidfVectorizer()
count_matrix = tfidf.fit_transform(movies['genres'])


# Nearest Neighbors model
nn = NearestNeighbors(n_neighbors=6, metric='cosine')
nn.fit(count_matrix)


# Recommendation function
def clean_title_ignore_year(title):
    title = title.lower().strip()
    title = re.sub(r'[^a-z0-9 ]', '', title)       # remove punctuation
    title = re.sub(r'\(\d{4}\)', '', title)        # remove year
    title = title.strip()
    return title

def get_recommendations(movie_title):
    result = process.extractOne(movie_title, movies['title'])
    if result is None:
        print("❌ Movie not found in dataset.")
        return
    match, score = result[0], result[1]
    if score < 60:
        print("❌ Movie not found in dataset.")
        return
    idx = movies[movies['title'] == match].index[0]
    distances, indices = nn.kneighbors(count_matrix[idx])
    print(f"\n🎬 Movies similar to '{movies.iloc[idx].title}':\n")
    for i in indices[0][1:]:
        print("➡️", movies.iloc[i].title)


print("=== Testing Block ===")
get_recommendations("Toy Story")
get_recommendations("Avatar")


=== Testing Block ===

🎬 Movies similar to 'Toy Story (1995)':

➡️ Shrek the Third (2007)
➡️ Tale of Despereaux, The (2008)
➡️ Antz (1998)
➡️ The Magic Crystal (2011)
➡️ Puss in Book: Trapped in an Epic Tale (2017)

🎬 Movies similar to 'Avatar (2009)':

➡️ The Hunger Games: Catching Fire (2013)
➡️ Ender's Game (2013)
➡️ John Carter (2012)
➡️ After Earth (2013)
➡️ Star Wars: Episode II - Attack of the Clones (2002)


In [21]:
print("\n=== Interactive Block ===")
movie = input("Enter a movie title to get recommendations: ")
get_recommendations(movie)



=== Interactive Block ===
Enter a movie title to get recommendations: batman begins

🎬 Movies similar to 'Batman Begins (2005)':

➡️ Need for Speed (2014)
➡️ Dark Knight, The (2008)
➡️ Good Day to Die Hard, A (2013)
➡️ Eagle Eye (2008)
➡️ Fast & Furious 6 (Fast and the Furious 6, The) (2013)
